In [ ]:
import numpy as np
import pandas as pd
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

# 1. Load data & generate fitur
raw = pd.read_csv("data/ohlcv.csv", parse_dates=["Date"], index_col="Date").sort_index()
df_feat, feature_cols = add_features(raw)   # fungsi add_features dari kode kamu

# 2. Load model
model = PPO.load("models/ppo_trading_model.zip", device="auto")

# Kalau training pakai VecNormalize, load juga statistiknya (jangan skip!)
vecnorm = None
# vecnorm = VecNormalize.load("models/vecnormalize.pkl", DummyVecEnv([lambda: None]))
# vecnorm.training = False
# vecnorm.norm_reward = False

# 3. Buat observasi & predict
def predict_action(row: pd.Series, deterministic: bool = True) -> int:
    obs = row[feature_cols].to_numpy(dtype=np.float32)
    if vecnorm is not None:
        obs = vecnorm.normalize_obs(obs.reshape(1, -1)).flatten()
    action, _ = model.predict(obs, deterministic=deterministic)
    return int(action)

# 4. Loop inference per-bar
df_feat["action"] = [predict_action(row) for _, row in df_feat.iterrows()]
df_feat["signal"] = df_feat["action"].map({0: "flat", 1: "long", 2: "short"})

In [1]:
import MetaTrader5 as mt5
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
mt5.__version__

'5.0.6147'

In [3]:
mt5.initialize()

True

In [4]:
def predict_signals(df, features, model, vecnorm):
    directions, sl_indices, tp_indices = [], [], []
    position_state = np.zeros(6, dtype=np.float32)
    current_position = None  # track open position

    for idx, row in df.iterrows():
        market_obs = row[features].to_numpy(dtype=np.float32)
        
        # Update position_state berdasarkan posisi yang sedang terbuka
        if current_position is not None:
            close = float(row["Close"])
            atr = max(float(row.get("atr", 1.0)), 1e-12)
            p = current_position
            unrealized = (close - p["entry_price"]) * p["units"] * p["direction"]
            position_state = np.array([
                p["direction"],
                unrealized / max(p["risk_cash"], 1e-12),
                min(p["bars_in_trade"] / 100.0, 10.0),
                ((p["tp"] - close) * p["direction"]) / atr,
                ((close - p["sl"]) * p["direction"]) / atr,
                p["tp_r"],
            ], dtype=np.float32)
        else:
            position_state = np.zeros(6, dtype=np.float32)

        obs = np.concatenate([market_obs, position_state])
        obs_normalized = vecnorm.normalize_obs(obs.reshape(1, -1))
        action, _ = model.predict(obs_normalized, deterministic=True)

        directions.append(action[0][0])
        sl_indices.append(action[0][1])
        tp_indices.append(action[0][2])
    
    return directions, sl_indices, tp_indices


In [5]:
from stable_baselines3 import PPO
model = PPO.load("../model/2_return_43%_1H/ppo_xauusd.zip", device="cpu")

In [6]:
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

# 1. Definisikan DummyEnv dengan observation space berukuran 31
class DummyEnv(gym.Env):
    def __init__(self):
        # 31 berasal dari: 25 market features + 6 position state features
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(31,), dtype=np.float32)
        self.action_space = spaces.MultiDiscrete([3, 3, 4])

# 2. Bungkus DummyEnv ke dalam VecEnv
dummy_env = DummyVecEnv([lambda: DummyEnv()])

# 3. Load file statistik vecnorm.pkl yang tersimpan saat training
# Sesuaikan path ini dengan lokasi file ppo_xauusd_vecnorm.pkl Anda
vecnorm_path = "../model/2_return_43%_1H/ppo_xauusd_vecnorm.pkl"
vecnorm = VecNormalize.load(vecnorm_path, dummy_env)

# Set agar vecnorm berada pada mode evaluasi (tidak melakukan update statistik baru)
vecnorm.training = False
vecnorm.norm_reward = False


In [7]:
from datetime import datetime

dtfrom = datetime(2026, 1, 1)
dt_to = datetime(2026, 3, 1)
data = mt5.copy_rates_range("XAUUSD", mt5.TIMEFRAME_H1, dtfrom, dt_to)
data = pd.DataFrame(data)
data['time'] = pd.to_datetime(data['time'], unit="s", )
data.columns = [c.capitalize() for c in data.columns]

In [21]:
data = mt5.copy_rates_from_pos("XAUUSD", mt5.TIMEFRAME_H1, 0, 700)
data = pd.DataFrame(data)
data['time'] = pd.to_datetime(data['time'], unit="s", )
data.columns = [c.capitalize() for c in data.columns]

In [11]:
import sys, os
sys.path.append(os.path.abspath(".."))

from reinforcement_learning.features import add_features

df_feat, feature_cols = add_features(data)

In [12]:
dirs, sls, tps = predict_signals(df_feat, feature_cols, model, vecnorm)

In [13]:
df_feat["pred_direction"] = dirs
df_feat["pred_sl_idx"] = sls
df_feat["pred_tp_idx"] = tps
df_feat["signal"] = df_feat["pred_direction"].map({0: "flat", 1: "long", 2: "short"})
df_feat[df_feat["pred_direction"] != 0][["Time", "Close", "signal", "pred_sl_idx", "pred_tp_idx"]].head(10)

,Time,Close,signal,pred_sl_idx,pred_tp_idx
250,2026-01-16 14:00:00,4601.87,long,2,2
251,2026-01-16 15:00:00,4601.88,long,2,2
253,2026-01-16 17:00:00,4571.27,long,2,2
254,2026-01-16 18:00:00,4591.33,long,2,1
255,2026-01-16 19:00:00,4581.48,long,2,1
256,2026-01-16 20:00:00,4582.71,long,2,1
257,2026-01-16 21:00:00,4590.59,long,2,0
258,2026-01-16 22:00:00,4581.92,long,2,1
262,2026-01-19 03:00:00,4657.38,long,2,3
263,2026-01-19 04:00:00,4662.42,long,2,3


In [14]:
df_feat.signal.value_counts()

signal
long    612
flat     80
Name: count, dtype: int64

In [15]:
check = df_feat[df_feat['signal'] == "flat"]
check.head()

,Time,Open,High,Low,Close,Tick_volume,Spread,Real_volume,atr,ema20,...,dow_sin,dow_cos,session_asia,session_london,session_newyork,session_london_ny_overlap,pred_direction,pred_sl_idx,pred_tp_idx,signal
252,2026-01-16 16:00:00,4601.88,4614.76,4596.18,4607.54,34697,4,0,15.117443,4605.201904,...,0.433884,-0.900969,1,0,0,0,0,2,1,flat
259,2026-01-16 23:00:00,4581.92,4598.56,4581.90,4596.42,15496,4,0,19.038719,4595.546187,...,0.433884,-0.900969,1,0,0,0,0,2,0,flat
260,2026-01-19 01:00:00,4621.06,4690.69,4621.06,4672.52,26843,4,0,24.412382,4602.877026,...,0.433884,-0.900969,1,0,0,0,0,2,0,flat
261,2026-01-19 02:00:00,4672.52,4682.03,4661.18,4675.19,25795,4,0,24.157926,4609.763976,...,0.433884,-0.900969,1,0,0,0,0,2,3,flat
285,2026-01-20 05:00:00,4668.68,4681.49,4666.11,4680.16,16641,4,0,13.245570,4664.246470,...,0.433884,-0.900969,1,0,0,0,0,2,1,flat


In [16]:
pos = mt5.positions_get(symbol="XAUUSD")
pos

()

In [22]:
mt5.account_info().equity

100000.0

In [5]:
price = mt5.symbol_info_tick("XAUUSD")

In [6]:
price

Tick(time=1788247702, bid=4431.01, ask=4431.41, last=0.0, volume=0, time_msc=1788247702407, flags=1028, volume_real=0.0)